# Complete Model Comparison Table

Produces the two-section LaTeX table:
- **Top (varying seed):** mean ± std over 10 seeds × 5 folds = 50 values per model.  
  Wilcoxon signed-rank (one-sided, EEG-TACT > baseline, p < 0.05) → `\cellcolor{lightgray}`.
- **Bottom (best seed):** seed 123, mean ± std over 5 folds.  
  McNemar exact test on paired subject predictions → `\cellcolor{lightgray}` in Odds Ratio row.

Output saved to `../results/evaluation/complete_model_comparison_table.tex`

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy.stats import wilcoxon
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, roc_auc_score
)
from statsmodels.stats.contingency_tables import mcnemar

HYP  = Path('../results/hypothesis tests')
BASE = Path('../results')

# The 10 seeds used in every experiment
SEEDS = {7, 42, 99, 123, 314, 456, 789, 2024, 2718, 5000}

# Column order in the LaTeX table
MODELS   = ['EEGNet', 'ShallowConvNet', 'CNN-LSTM', 'MultiStream',
            'EEGConformer', 'IMCBGT', 'TGARNet', 'EEGFormer']
HEADERS  = ['EEGNet', 'ShallowConvNet', 'CNN-LSTM', 'Multi-Stream',
            'EEGConformer', 'IM-CBGT', 'T-GARNet', 'EEG-TACT']

METRICS  = ['accuracy', 'recall', 'precision', 'specificity', 'auc']
REF      = 'EEGFormer'

print('Imports OK')

Imports OK


## Part 1 — Varying seed (50 folds per model)

In [2]:
# --- Infer class prevalence per fold from CNN-LSTM (has direct recall/precision) ---
# accuracy  = p*recall + (1-p)*specificity
# bal_acc   = (recall + specificity) / 2
# → recall  = (accuracy - 2*(1-p)*bal) / (2p-1)   (when p != 0.5)
# Rearranging for p:  p = (accuracy - specificity) / (recall - specificity)

_cnn = pd.read_csv(HYP / 'cnn_lstm_eegnet_10seeds' / 'all_fold_results.csv')
_cnn = _cnn[_cnn.seed.astype(int).isin(SEEDS)].copy()

fold_prev = {}
for fold_id, g in _cnn.groupby('fold'):
    a  = g['accuracy'].astype(float).values
    b  = g['balanced_acc'].astype(float).values
    r  = g['recall'].astype(float).values
    sp = 2.0 * b - r          # specificity = 2*bacc - recall
    dn = r - sp                # denominator = recall - specificity
    ok = np.abs(dn) > 1e-9
    fold_prev[int(fold_id)] = float(np.nanmean((a[ok] - sp[ok]) / dn[ok]))

print('ADHD prevalence per fold (should be ~0.46–0.63):')
for k, v in sorted(fold_prev.items()):
    print(f'  fold {k}: {v:.4f}')

ADHD prevalence per fold (should be ~0.46–0.63):
  fold 0: 0.4583
  fold 1: 0.5417
  fold 2: 0.4583
  fold 3: 0.6250
  fold 4: 0.4583


In [3]:

# --- Load 50 (seed × fold) rows per model ---

def load_fold_metrics(folder, has_direct_rp=False):
    """Read all_fold_results.csv, reconstruct recall/prec/spec when not stored directly."""
    df = pd.read_csv(HYP / folder / 'all_fold_results.csv')
    df = df[df.seed.astype(int).isin(SEEDS)].copy()
    df['seed'] = df['seed'].astype(int)
    df['fold'] = df['fold'].astype(int)

    rows = []
    for _, row in df.iterrows():
        a   = float(row['accuracy'])
        b   = float(row['balanced_acc'])
        f1v = float(row['f1'])
        au  = float(row['auc'])
        fid = int(row['fold'])

        if has_direct_rp:
            rec  = float(row['recall'])
            prec = float(row['precision'])
            spec = float(np.clip(2.0 * b - rec, 0.0, 1.0))
        else:
            p   = fold_prev[fid]
            dn  = 2.0 * p - 1.0
            if abs(dn) < 1e-9:
                rec = prec = spec = np.nan
            else:
                rec  = float(np.clip((a - 2.0 * (1.0 - p) * b) / dn, 0.0, 1.0))
                spec = float(np.clip(2.0 * b - rec, 0.0, 1.0))
                dn2  = 2.0 * rec - f1v
                prec = float(np.clip(f1v * rec / dn2, 0.0, 1.0)) if abs(dn2) > 1e-9 else np.nan

        rows.append({'seed': int(row['seed']), 'fold': fid,
                     'accuracy': a, 'recall': rec, 'precision': prec,
                     'specificity': spec, 'auc': au})
    return pd.DataFrame(rows)


def load_conformer_fold_metrics():
    """EEGConformer: fold_results.json per seed + AUC from trial_predictions.csv.
    Seed is read from the folder name (seed_1104 has a different JSON schema – skipped
    because 1104 is not in SEEDS).
    """
    conf_dir = HYP / 'eegconformer_10seeds'

    # AUC from subject-level predicted probabilities
    auc_map = {}
    for sd in sorted(conf_dir.glob('seed_*')):
        sv = int(sd.name.split('_')[1])   # seed from folder name
        if sv not in SEEDS:
            continue
        tp = sd / 'trial_predictions.csv'
        if not tp.exists():
            continue
        t    = pd.read_csv(tp)
        subj = t[['fold', 'subject', 'true_label', 'subject_mean_prob']].drop_duplicates()
        for fv, fg in subj.groupby('fold'):
            yt = fg['true_label'].astype(int).values
            yp = fg['subject_mean_prob'].astype(float).values
            if np.unique(yt).size >= 2:
                auc_map[(sv, int(fv))] = float(roc_auc_score(yt, yp))

    rows = []
    for sd in sorted(conf_dir.glob('seed_*')):
        sv = int(sd.name.split('_')[1])   # seed from folder name
        if sv not in SEEDS:               # skip seed_1104 (different JSON schema)
            continue
        jp = sd / 'fold_results.json'
        if not jp.exists():
            continue
        entries = json.loads(jp.read_text())
        for e in entries:
            fv  = int(e['fold'])
            a   = float(e['test_subject_acc'])
            b   = float(e['test_subject_bacc'])
            f1v = float(e['test_subject_f1'])
            p   = fold_prev[fv]
            dn  = 2.0 * p - 1.0
            if abs(dn) < 1e-9:
                rec = prec = spec = np.nan
            else:
                rec  = float(np.clip((a - 2.0 * (1.0 - p) * b) / dn, 0.0, 1.0))
                spec = float(np.clip(2.0 * b - rec, 0.0, 1.0))
                dn2  = 2.0 * rec - f1v
                prec = float(np.clip(f1v * rec / dn2, 0.0, 1.0)) if abs(dn2) > 1e-9 else np.nan
            rows.append({'seed': sv, 'fold': fv, 'accuracy': a,
                         'recall': rec, 'precision': prec,
                         'specificity': spec, 'auc': auc_map.get((sv, fv), np.nan)})
    return pd.DataFrame(rows)


SOURCES = {
    'EEGNet':        ('eegnet_10seeds',         False),
    'ShallowConvNet':('shallowconvnet_10seeds',  False),
    'CNN-LSTM':      ('cnn_lstm_eegnet_10seeds', True),
    'MultiStream':   ('multistream_10seeds',     False),
    'IMCBGT':        ('imcbgt_10seeds',          True),
    'TGARNet':       ('tgarnet_10seeds',         False),
    'EEGFormer':     ('eegformer_10seeds',       False),
}

fold_data = {name: load_fold_metrics(folder, direct) for name, (folder, direct) in SOURCES.items()}
fold_data['EEGConformer'] = load_conformer_fold_metrics()

print('Rows per model (expect 50 each):')
for m in MODELS:
    df = fold_data[m]
    print(f'  {m:15s}: {len(df)} rows | {df.seed.nunique()} seeds × {df.fold.nunique()} folds')


Rows per model (expect 50 each):
  EEGNet         : 50 rows | 10 seeds × 5 folds
  ShallowConvNet : 50 rows | 10 seeds × 5 folds
  CNN-LSTM       : 50 rows | 10 seeds × 5 folds
  MultiStream    : 50 rows | 10 seeds × 5 folds
  EEGConformer   : 50 rows | 10 seeds × 5 folds
  IMCBGT         : 50 rows | 10 seeds × 5 folds
  TGARNet        : 50 rows | 10 seeds × 5 folds
  EEGFormer      : 50 rows | 10 seeds × 5 folds


In [23]:
# --- Wilcoxon signed-rank: one-sided EEGFormer > baseline, per metric ---

ref = fold_data[REF][['seed', 'fold'] + METRICS].rename(
    columns={m: m + '_ref' for m in METRICS}
)

w_sig  = {}   # (metric, model) -> bool
w_pval = {}   # (metric, model) -> float

for model in [m for m in MODELS if m != REF]:
    cmp = fold_data[model][['seed', 'fold'] + METRICS].rename(
        columns={m: m + '_cmp' for m in METRICS}
    )
    merged = ref.merge(cmp, on=['seed', 'fold'], how='inner')

    for metric in METRICS:
        pair = merged[['seed', 'fold', metric + '_ref', metric + '_cmp']].dropna()
        if len(pair) < 2:
            w_sig[(metric, model)]  = False
            w_pval[(metric, model)] = np.nan
            continue
        a = pair[metric + '_ref'].values
        b = pair[metric + '_cmp'].values
        diff = a - b
        if np.allclose(diff, 0.0):
            p = 1.0
        else:
            result = wilcoxon(a, b, alternative='greater',
                              zero_method='wilcox', mode='auto')
            p = float(result.pvalue)
        w_sig[(metric, model)]  = p < 0.05
        w_pval[(metric, model)] = p

baselines = [m for m in MODELS if m != REF]
pval_table = pd.DataFrame(
    {b: {mt: round(w_pval.get((mt, b), np.nan), 4) for mt in METRICS}
     for b in baselines}
).T
pval_table.index.name = 'baseline'
print('Wilcoxon p-values (one-sided EEG-TACT > baseline):')
print('Shaded in table when p < 0.05')
display(pval_table.T)

Wilcoxon p-values (one-sided EEG-TACT > baseline):
Shaded in table when p < 0.05


baseline,EEGNet,ShallowConvNet,CNN-LSTM,MultiStream,EEGConformer,IMCBGT,TGARNet
accuracy,0.0000,0.0003,0.5334,0.0,0.0000,0.0,0.0000
recall,0.0030,0.0220,0.9559,0.0,0.1424,0.0,0.9898
precision,0.0022,0.2581,0.3173,0.0,0.0000,0.0,0.0000
specificity,0.0627,0.2131,0.1219,0.0,0.0000,0.0,0.0000
auc,0.0059,0.3595,0.9865,0.0,0.0001,0.0,0.0000


In [13]:
# --- Varying-seed display statistics ---
# CNN-LSTM, IMCBGT, EEGConformer: fold-level std (std of all 50 values)
# All others: seed-level std (std of 10 per-seed means → between-seed variability)

FOLD_LEVEL = {'CNN-LSTM', 'IMCBGT', 'EEGConformer'}

var_stats = {}   # (model, metric) -> (mean_pct, std_pct)
for model in MODELS:
    df = fold_data[model]
    for metric in METRICS:
        vals = df[metric].dropna().astype(float)
        if model in FOLD_LEVEL:
            mu  = 100.0 * vals.mean()
            std = 100.0 * vals.std(ddof=1)
        else:
            seed_means = df.groupby('seed')[metric].mean().dropna()
            mu  = 100.0 * seed_means.mean()
            std = 100.0 * seed_means.std(ddof=1)
        var_stats[(model, metric)] = (float(mu), float(std))

rows = []
for model in MODELS:
    row = {'model': model}
    for metric in METRICS:
        mu, std = var_stats[(model, metric)]
        row[metric] = f'{mu:.1f} ± {std:.1f}' if not np.isnan(mu) else '—'
    rows.append(row)
print('Varying-seed statistics (mean ± std %):')
display(pd.DataFrame(rows).set_index('model').T)

Varying-seed statistics (mean ± std %):


model,EEGNet,ShallowConvNet,CNN-LSTM,MultiStream,EEGConformer,IMCBGT,TGARNet,EEGFormer
accuracy,80.7 ± 1.3,78.7 ± 3.2,84.2 ± 7.4,69.3 ± 2.9,77.2 ± 8.2,70.6 ± 9.6,74.8 ± 2.9,84.2 ± 1.8
recall,88.8 ± 2.0,84.9 ± 12.2,93.8 ± 8.9,79.3 ± 3.6,89.8 ± 12.8,80.0 ± 12.0,96.2 ± 1.3,92.8 ± 2.6
precision,77.7 ± 1.2,78.9 ± 5.3,79.9 ± 7.7,67.9 ± 3.3,73.6 ± 8.5,68.9 ± 9.6,68.3 ± 2.1,80.0 ± 1.6
specificity,74.2 ± 1.7,74.0 ± 9.2,75.1 ± 11.0,60.2 ± 4.7,65.3 ± 13.9,61.6 ± 15.0,53.4 ± 6.0,76.2 ± 2.2
auc,89.2 ± 1.0,90.3 ± 0.8,92.0 ± 5.6,76.3 ± 2.1,86.5 ± 7.2,78.9 ± 5.0,78.6 ± 3.8,90.5 ± 1.3


In [22]:
# --- Average accuracy rank (7 original models, EEGConformer excluded) ---
# Consistent with scripts/friedman_test.py and scripts/wilcoxon_test.py

RANK_MODELS = [m for m in MODELS]

rank_df = pd.DataFrame(
    {m: fold_data[m].set_index(['seed', 'fold'])['accuracy'] for m in RANK_MODELS}
).dropna(how='any')

avg_rank = rank_df.rank(axis=1, ascending=False).mean()
print(f'Average accuracy rank over {len(RANK_MODELS)} models (lower = better):')
display(avg_rank.sort_values().to_frame('avg_rank').round(2))

Average accuracy rank over 8 models (lower = better):


,avg_rank
EEGFormer,2.61
CNN-LSTM,2.65
EEGNet,3.85
ShallowConvNet,4.12
EEGConformer,4.78
TGARNet,5.29
IMCBGT,6.10
MultiStream,6.60


## Part 2 — Best seed (seed 123, 5 folds)

In [24]:
# --- Load subject-level majority-vote predictions for seed 123 ---

def subject_metrics_from_csv(path):
    """Return per-fold (accuracy, recall, precision, specificity, auc) from trial_probs CSV."""
    df = pd.read_csv(path)
    subj = df[['fold', 'subject', 'true_label',
               'subject_majority_pred', 'subject_mean_prob']].drop_duplicates()
    rows = []
    for fold, g in subj.groupby('fold'):
        yt = g['true_label'].astype(int).values
        yp = g['subject_majority_pred'].astype(int).values
        ys = g['subject_mean_prob'].astype(float).values
        tn = int(((yt == 0) & (yp == 0)).sum())
        fp = int(((yt == 0) & (yp == 1)).sum())
        rows.append({
            'fold':        fold,
            'accuracy':    accuracy_score(yt, yp),
            'recall':      recall_score(yt, yp, zero_division=0),
            'precision':   precision_score(yt, yp, zero_division=0),
            'specificity': tn / (tn + fp) if (tn + fp) > 0 else np.nan,
            'auc':         roc_auc_score(yt, ys) if np.unique(yt).size >= 2 else np.nan,
        })
    return pd.DataFrame(rows)


BEST_SEED_FILES = {
    'EEGNet':        HYP / 'eegnet_10seeds'          / 'all_trial_probs.csv',
    'ShallowConvNet':HYP / 'shallowconvnet_10seeds'   / 'all_trial_probs.csv',
    'CNN-LSTM':      HYP / 'cnn_lstm_eegnet_10seeds'  / 'all_trial_probs.csv',
    'MultiStream':   HYP / 'multistream_10seeds'      / 'all_trial_probs.csv',
    'IMCBGT':        HYP / 'imcbgt_10seeds'           / 'all_trial_probs.csv',
    'TGARNet':       HYP / 'tgarnet_10seeds'          / 'all_trial_probs.csv',
    'EEGFormer':     BASE / 'eegformer_5fold_run'     / 'all_trial_probs.csv',
    'EEGConformer':  HYP / 'eegconformer_10seeds'     / 'seed_123' / 'trial_predictions.csv',
}

best_fold = {name: subject_metrics_from_csv(path) for name, path in BEST_SEED_FILES.items()}

# Summary: mean ± std over 5 folds
best_stats = {}   # (model, metric) -> (mean_pct, std_pct)
for model in MODELS:
    df = best_fold[model]
    for metric in METRICS:
        vals = df[metric].dropna().astype(float)
        best_stats[(model, metric)] = (100.0 * vals.mean(), 100.0 * vals.std(ddof=1))

rows = []
for model in MODELS:
    row = {'model': model}
    for metric in METRICS:
        mu, std = best_stats[(model, metric)]
        row[metric] = f'{mu:.1f} ± {std:.1f}' if not np.isnan(mu) else '—'
    rows.append(row)
print('Best-seed statistics (seed 123, mean ± std over 5 folds):')
display(pd.DataFrame(rows).set_index('model'))

Best-seed statistics (seed 123, mean ± std over 5 folds):


,accuracy,recall,precision,specificity,auc
model,,,,,
EEGNet,79.2 ± 9.3,88.4 ± 16.5,76.2 ± 8.5,71.9 ± 9.7,87.2 ± 6.3
ShallowConvNet,80.8 ± 3.7,91.1 ± 9.0,76.7 ± 8.2,71.9 ± 8.0,91.2 ± 3.4
CNN-LSTM,80.0 ± 10.8,93.3 ± 14.9,76.0 ± 11.3,68.8 ± 17.6,92.1 ± 7.2
MultiStream,73.3 ± 5.6,79.1 ± 13.4,74.1 ± 12.8,68.2 ± 19.1,78.0 ± 8.7
EEGConformer,74.2 ± 9.0,85.2 ± 23.5,73.2 ± 10.5,64.7 ± 20.1,84.6 ± 8.4
IMCBGT,69.2 ± 9.6,77.4 ± 11.0,68.4 ± 10.4,61.0 ± 18.7,78.5 ± 4.8
TGARNet,70.0 ± 14.3,96.8 ± 4.4,64.8 ± 12.2,43.2 ± 25.6,72.1 ± 14.2
EEGFormer,87.5 ± 5.1,96.0 ± 8.9,82.8 ± 4.0,79.6 ± 3.5,89.9 ± 5.2


In [25]:
# --- McNemar exact test (two-sided) on subject-level correctness ---
# Pairs subjects across folds: EEGFormer (seed 123) vs each baseline (seed 123)

def subject_correctness(path):
    df = pd.read_csv(path)
    subj = df[['fold', 'subject', 'true_label', 'subject_majority_pred']].drop_duplicates()
    subj = subj.copy()
    subj['correct'] = (subj['subject_majority_pred'].astype(int)
                       == subj['true_label'].astype(int)).astype(int)
    return subj[['fold', 'subject', 'correct']]


ref_correct = subject_correctness(BEST_SEED_FILES[REF])

mc_rows = []
for model in [m for m in MODELS if m != REF]:
    cmp_correct = subject_correctness(BEST_SEED_FILES[model])
    merged = ref_correct.merge(
        cmp_correct, on=['fold', 'subject'], suffixes=('_ref', '_cmp'), how='inner'
    )

    n11 = int(((merged.correct_ref == 1) & (merged.correct_cmp == 1)).sum())
    n10 = int(((merged.correct_ref == 1) & (merged.correct_cmp == 0)).sum())  # EEG-TACT wins
    n01 = int(((merged.correct_ref == 0) & (merged.correct_cmp == 1)).sum())  # baseline wins
    n00 = int(((merged.correct_ref == 0) & (merged.correct_cmp == 0)).sum())

    # statsmodels mcnemar: table = [[both correct, ref-only], [baseline-only, both wrong]]
    result = mcnemar([[n11, n10], [n01, n00]], exact=True, correction=False)

    mc_rows.append({
        'model':    model,
        'n_pairs':  len(merged),
        'n10 (EEG-TACT wins)':  n10,
        'n01 (baseline wins)':  n01,
        'OR (n10/n01)': f'{n10}/{n01}',
        'p_value':  float(result.pvalue),
        'sig (p<0.05)': float(result.pvalue) < 0.05,
    })

mc_df = pd.DataFrame(mc_rows)
print('McNemar exact test (two-sided) — EEG-TACT vs each baseline:')
display(mc_df.to_string(index=False))

McNemar exact test (two-sided) — EEG-TACT vs each baseline:


'         model  n_pairs  n10 (EEG-TACT wins)  n01 (baseline wins) OR (n10/n01)  p_value  sig (p<0.05)\n        EEGNet      120                   11                    1         11/1 0.006348          True\nShallowConvNet      120                   11                    3         11/3 0.057373         False\n      CNN-LSTM      120                   11                    2         11/2 0.022461          True\n   MultiStream      120                   24                    7         24/7 0.003327          True\n  EEGConformer      120                   19                    3         19/3 0.000855          True\n        IMCBGT      120                   28                    6         28/6 0.000195          True\n       TGARNet      120                   23                    2         23/2 0.000019          True'

## Part 3 — Assemble LaTeX table

In [9]:
METRIC_LABELS = {
    'accuracy':    r'Accuracy (\%)',
    'recall':      r'Recall (\%)',
    'precision':   r'Precision (\%)',
    'specificity': r'Specificity (\%)',
    'auc':         r'AUC (\%)',
}
YEAR   = {'EEGNet':'2017','ShallowConvNet':'2017','CNN-LSTM':'2022',
          'MultiStream':'2023','EEGConformer':'2023','IMCBGT':'2024',
          'TGARNet':'2025','EEGFormer':'2026'}
PARAMS = {'EEGNet':'1{,}746','ShallowConvNet':'36{,}522','CNN-LSTM':'9{,}052',
          'MultiStream':'574{,}082','EEGConformer':'789{,}506',
          'IMCBGT':'1{,}195{,}266','TGARNet':'9{,}071','EEGFormer':'7{,}322'}

SEP = '\n& '
END = r'\\'


def best_in_row(stats_dict, metric):
    vals = [round(stats_dict[(m, metric)][0], 1) for m in MODELS
            if not np.isnan(stats_dict[(m, metric)][0])]
    return max(vals) if vals else np.nan


def fmt(mu, std, bold, shade):
    if np.isnan(mu):
        return ''
    inner = f'{mu:.1f} \\pm {std:.1f}'
    if bold:
        inner = f'\\mathbf{{{inner}}}'
    body = f'${{{inner}}}$'
    if shade:
        body = f'\\cellcolor{{lightgray}}{body}'
    return body


def fmt_var(model, metric):
    mu, std = var_stats[(model, metric)]
    best  = best_in_row(var_stats, metric)
    shade = (model != REF) and w_sig.get((metric, model), False)
    return fmt(mu, std, round(mu, 1) == best, shade)


def fmt_best(model, metric):
    mu, std = best_stats[(model, metric)]
    best = best_in_row(best_stats, metric)
    return fmt(mu, std, round(mu, 1) == best, False)


def fmt_or(model):
    if model == REF:
        return '--'
    row = mc_df[mc_df.model == model].iloc[0]
    body = f'${{{row["OR (n10/n01)"]}}}\'$'
    # fix: correct f-string
    body = '${{{}}}$'.format(row['OR (n10/n01)'])
    if row['sig (p<0.05)']:
        body = f'\\cellcolor{{lightgray}}{body}'
    return body


# Build rank cells (EEGConformer left blank)
best_rank = avg_rank.min()
rank_cells = []
for m in MODELS:
    if m == 'EEGConformer':
        rank_cells.append('')
    else:
        rv = avg_rank[m]
        if abs(rv - best_rank) < 1e-9:
            rank_cells.append(f'$\\mathbf{{{rv:.2f}}}$')
        else:
            rank_cells.append(f'${rv:.2f}$')


# Assemble
lines = [
    r'\begin{tabular}{lcccccccc}',
    r'\toprule',
    r'\textbf{Model}' + SEP + SEP.join(f'\\textbf{{{h}}}' for h in HEADERS) + END,
    r'\midrule',
    'Year'       + SEP + SEP.join(YEAR[m]   for m in MODELS) + END,
    'Parameters' + SEP + SEP.join(PARAMS[m] for m in MODELS) + END,
    r'\midrule',
    r'\multicolumn{9}{c}{\textbf{Varying seed (average over 50 folds)}} \\',
    r'\midrule',
]

for metric in METRICS:
    lines.append(METRIC_LABELS[metric] + SEP
                 + SEP.join(fmt_var(m, metric) for m in MODELS) + END)

lines.append('Average Rank' + SEP + SEP.join(rank_cells) + END)

lines += [
    r'\midrule',
    r'\multicolumn{9}{c}{\textbf{Best seed (average over 5 folds)}} \\',
    r'\midrule',
]

for metric in METRICS:
    lines.append(METRIC_LABELS[metric] + SEP
                 + SEP.join(fmt_best(m, metric) for m in MODELS) + END)

lines.append('Odds ratio' + SEP + SEP.join(fmt_or(m) for m in MODELS) + END)
lines += [r'\bottomrule', r'\end{tabular}']

latex_out = '\n'.join(lines)

out_path = Path('../results/evaluation/complete_model_comparison_table.tex')
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(latex_out)
print(f'Saved to: {out_path.resolve()}')
display(Markdown(f'```latex\n{latex_out}\n```'))

Saved to: /var/home/automatica/Documents/GitHub/EEGFormer/results/evaluation/complete_model_comparison_table.tex


```latex
\begin{tabular}{lcccccccc}
\toprule
\textbf{Model}
& \textbf{EEGNet}
& \textbf{ShallowConvNet}
& \textbf{CNN-LSTM}
& \textbf{Multi-Stream}
& \textbf{EEGConformer}
& \textbf{IM-CBGT}
& \textbf{T-GARNet}
& \textbf{EEG-TACT}\\
\midrule
Year
& 2017
& 2017
& 2022
& 2023
& 2023
& 2024
& 2025
& 2026\\
Parameters
& 1{,}746
& 36{,}522
& 9{,}052
& 574{,}082
& 789{,}506
& 1{,}195{,}266
& 9{,}071
& 7{,}322\\
\midrule
\multicolumn{9}{c}{\textbf{Varying seed (average over 50 folds)}} \\
\midrule
Accuracy (\%)
& \cellcolor{lightgray}${80.7 \pm 1.3}$
& \cellcolor{lightgray}${78.7 \pm 3.2}$
& ${\mathbf{84.2 \pm 7.4}}$
& \cellcolor{lightgray}${69.3 \pm 2.9}$
& \cellcolor{lightgray}${77.2 \pm 8.2}$
& \cellcolor{lightgray}${70.6 \pm 9.6}$
& \cellcolor{lightgray}${74.8 \pm 2.9}$
& ${\mathbf{84.2 \pm 1.8}}$\\
Recall (\%)
& \cellcolor{lightgray}${88.8 \pm 2.0}$
& \cellcolor{lightgray}${84.9 \pm 12.2}$
& ${93.8 \pm 8.9}$
& \cellcolor{lightgray}${79.3 \pm 3.6}$
& ${89.8 \pm 12.8}$
& \cellcolor{lightgray}${80.0 \pm 12.0}$
& ${\mathbf{96.2 \pm 1.3}}$
& ${92.8 \pm 2.6}$\\
Precision (\%)
& \cellcolor{lightgray}${77.7 \pm 1.2}$
& ${78.9 \pm 5.3}$
& ${79.9 \pm 7.7}$
& \cellcolor{lightgray}${67.9 \pm 3.3}$
& \cellcolor{lightgray}${73.6 \pm 8.5}$
& \cellcolor{lightgray}${68.9 \pm 9.6}$
& \cellcolor{lightgray}${68.3 \pm 2.1}$
& ${\mathbf{80.0 \pm 1.6}}$\\
Specificity (\%)
& ${74.2 \pm 1.7}$
& ${74.0 \pm 9.2}$
& ${75.1 \pm 11.0}$
& \cellcolor{lightgray}${60.2 \pm 4.7}$
& \cellcolor{lightgray}${65.3 \pm 13.9}$
& \cellcolor{lightgray}${61.6 \pm 15.0}$
& \cellcolor{lightgray}${53.4 \pm 6.0}$
& ${\mathbf{76.2 \pm 2.2}}$\\
AUC (\%)
& \cellcolor{lightgray}${89.2 \pm 1.0}$
& ${90.3 \pm 0.8}$
& ${\mathbf{92.0 \pm 5.6}}$
& \cellcolor{lightgray}${76.3 \pm 2.1}$
& \cellcolor{lightgray}${86.5 \pm 7.2}$
& \cellcolor{lightgray}${78.9 \pm 5.0}$
& \cellcolor{lightgray}${78.6 \pm 3.8}$
& ${90.5 \pm 1.3}$\\
Average Rank
& $3.53$
& $3.63$
& $2.44$
& $5.85$
& 
& $5.42$
& $4.70$
& $\mathbf{2.43}$\\
\midrule
\multicolumn{9}{c}{\textbf{Best seed (average over 5 folds)}} \\
\midrule
Accuracy (\%)
& ${79.2 \pm 9.3}$
& ${80.8 \pm 3.7}$
& ${80.0 \pm 10.8}$
& ${73.3 \pm 5.6}$
& ${74.2 \pm 9.0}$
& ${69.2 \pm 9.6}$
& ${70.0 \pm 14.3}$
& ${\mathbf{87.5 \pm 5.1}}$\\
Recall (\%)
& ${88.4 \pm 16.5}$
& ${91.1 \pm 9.0}$
& ${93.3 \pm 14.9}$
& ${79.1 \pm 13.4}$
& ${85.2 \pm 23.5}$
& ${77.4 \pm 11.0}$
& ${\mathbf{96.8 \pm 4.4}}$
& ${96.0 \pm 8.9}$\\
Precision (\%)
& ${76.2 \pm 8.5}$
& ${76.7 \pm 8.2}$
& ${76.0 \pm 11.3}$
& ${74.1 \pm 12.8}$
& ${73.2 \pm 10.5}$
& ${68.4 \pm 10.4}$
& ${64.8 \pm 12.2}$
& ${\mathbf{82.8 \pm 4.0}}$\\
Specificity (\%)
& ${71.9 \pm 9.7}$
& ${71.9 \pm 8.0}$
& ${68.8 \pm 17.6}$
& ${68.2 \pm 19.1}$
& ${64.7 \pm 20.1}$
& ${61.0 \pm 18.7}$
& ${43.2 \pm 25.6}$
& ${\mathbf{79.6 \pm 3.5}}$\\
AUC (\%)
& ${87.2 \pm 6.3}$
& ${91.2 \pm 3.4}$
& ${\mathbf{92.1 \pm 7.2}}$
& ${78.0 \pm 8.7}$
& ${84.6 \pm 8.4}$
& ${78.5 \pm 4.8}$
& ${72.1 \pm 14.2}$
& ${89.9 \pm 5.2}$\\
Odds ratio
& \cellcolor{lightgray}${11/1}$
& ${11/3}$
& \cellcolor{lightgray}${11/2}$
& \cellcolor{lightgray}${24/7}$
& \cellcolor{lightgray}${19/3}$
& \cellcolor{lightgray}${28/6}$
& \cellcolor{lightgray}${23/2}$
& --\\
\bottomrule
\end{tabular}
```